# 01 – Explorative Datenanalyse (EDA)
## ADSC21 Prüfungsleistung – Milwaukee Property Sales

**Ziel:** Verstehen der Datenstruktur, Bereinigung und aussagekräftige Visualisierungen als Grundlage für das Vorhersagemodell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 1. Daten laden

Datensatz: Milwaukee Property Sales 2002–2018 Master File von [Milwaukee Open Data](https://data.milwaukee.gov/dataset/property-sales-data).

In [ ]:
URL = 'https://data.milwaukee.gov/dataset/b6b2fbc1-2a11-45d4-9aba-3a5f7f9a0977/resource/b535e2cd-09dd-4aba-9c53-ee53f62cbf28/download/rp_sales.csv'

df_raw = pd.read_csv(URL, encoding='latin-1', low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Spaltennamen bereinigen
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(' ', '_')
print('Columns:', df_raw.columns.tolist())

## 2. Erste Übersicht & Datenbereinigung

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all').T

In [ ]:
# Fehlende Werte pro Spalte
missing = df_raw.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].to_string())

In [ ]:
# Zielvariable: sale_price (evtl. auch 'saleprice' je nach Dataset-Version)
price_col = 'sale_price' if 'sale_price' in df_raw.columns else 'saleprice'

df = df_raw.copy()

# Preis als numerisch erzwingen
df[price_col] = pd.to_numeric(df[price_col], errors='coerce')

# Zeilen entfernen: kein Preis, Preis <= 0 (Schenkungen, familienintere Transaktionen)
df = df[df[price_col] > 0].dropna(subset=[price_col])

# Extremwerte: oberes 1%-Quantil ausschließen (Luxusausreißer)
upper = df[price_col].quantile(0.99)
df = df[df[price_col] <= upper]

print(f'Zeilen nach Bereinigung: {len(df):,}')
print(f'Preisbereich: ${df[price_col].min():,.0f} – ${df[price_col].max():,.0f}')
print(f'Median Preis: ${df[price_col].median():,.0f}')

## 3. Visualisierungen
### 3.1 Verteilung des Verkaufspreises

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Originalverteilung
axes[0].hist(df[price_col] / 1000, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Verteilung Verkaufspreis', fontsize=13)
axes[0].set_xlabel('Verkaufspreis (Tsd. USD)')
axes[0].set_ylabel('Häufigkeit')
axes[0].axvline(df[price_col].median() / 1000, color='tomato', lw=2, label=f'Median: ${df[price_col].median()/1000:.0f}k')
axes[0].legend()

# Log-transformierte Verteilung
log_prices = np.log1p(df[price_col])
axes[1].hist(log_prices, bins=80, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_title('Verteilung log(Verkaufspreis + 1)', fontsize=13)
axes[1].set_xlabel('log(Preis + 1)')
axes[1].set_ylabel('Häufigkeit')

plt.tight_layout()
plt.savefig('../report/abb1_preisverteilung.png', bbox_inches='tight')
plt.show()
print('Abbildung 1 gespeichert.')

### 3.2 Numerische Merkmale & Korrelationen

In [ ]:
# Kandidaten für numerische Features
num_candidates = ['fin_sqft', 'year_built', 'bedrooms', 'bathrooms', 'lotsize', 'stories', price_col]
num_cols = [c for c in num_candidates if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Korrelationsmatrix – numerische Merkmale', fontsize=13)
plt.tight_layout()
plt.savefig('../report/abb2_korrelation.png', bbox_inches='tight')
plt.show()

### 3.3 Wohnfläche vs. Verkaufspreis

In [ ]:
if 'fin_sqft' in df.columns and 'style' in df.columns:
    top_styles = df['style'].value_counts().head(5).index
    df_plot = df[df['style'].isin(top_styles)].sample(min(5000, len(df)), random_state=42)

    fig, ax = plt.subplots(figsize=(11, 6))
    for style in top_styles:
        subset = df_plot[df_plot['style'] == style]
        ax.scatter(subset['fin_sqft'], subset[price_col] / 1000,
                   alpha=0.4, s=12, label=style)
    ax.set_xlabel('Fertiggestellte Wohnfläche (sqft)')
    ax.set_ylabel('Verkaufspreis (Tsd. USD)')
    ax.set_title('Wohnfläche vs. Verkaufspreis nach Gebäudestil', fontsize=13)
    ax.legend(title='Stil', fontsize=8)
    plt.tight_layout()
    plt.savefig('../report/abb3_flaeche_preis.png', bbox_inches='tight')
    plt.show()
elif 'fin_sqft' in df.columns:
    df.plot.scatter(x='fin_sqft', y=price_col, alpha=0.3, figsize=(10, 5))
    plt.title('Wohnfläche vs. Verkaufspreis')
    plt.show()

### 3.4 Verkaufspreise nach Gebäudestil (Top 8)

In [ ]:
if 'style' in df.columns:
    top8 = df['style'].value_counts().head(8).index
    df_box = df[df['style'].isin(top8)]

    fig, ax = plt.subplots(figsize=(13, 6))
    order = df_box.groupby('style')[price_col].median().sort_values(ascending=False).index
    sns.boxplot(data=df_box, x='style', y=price_col, order=order,
                palette='Set2', showfliers=False, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
    ax.set_title('Verkaufspreis nach Gebäudestil (Top 8)', fontsize=13)
    ax.set_xlabel('Gebäudestil')
    ax.set_ylabel('Verkaufspreis')
    plt.tight_layout()
    plt.savefig('../report/abb4_preis_stil.png', bbox_inches='tight')
    plt.show()

### 3.5 Zeitreihe: Median-Verkaufspreis nach Jahr

In [ ]:
year_col = next((c for c in df.columns if 'year' in c and 'built' not in c), None)
if year_col:
    df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
    yearly = df.groupby(year_col)[price_col].agg(['median', 'count'])
    yearly = yearly[yearly['count'] >= 20]  # mind. 20 Transaktionen

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()
    ax1.plot(yearly.index, yearly['median'] / 1000, 'b-o', ms=4, label='Median Preis')
    ax2.bar(yearly.index, yearly['count'], alpha=0.25, color='gray', label='Anzahl Verkäufe')
    ax1.set_ylabel('Median Verkaufspreis (Tsd. USD)', color='blue')
    ax2.set_ylabel('Anzahl Verkäufe', color='gray')
    ax1.set_title('Median-Verkaufspreis & Verkaufsvolumen nach Jahr', fontsize=13)
    ax1.set_xlabel('Jahr')
    plt.tight_layout()
    plt.savefig('../report/abb5_zeitreihe.png', bbox_inches='tight')
    plt.show()
else:
    print('Kein Jahres-Merkmal gefunden. Verfügbare Spalten:', df.columns.tolist())

## 4. Feature-Auswahl & Speichern des bereinigten Datensatzes

In [ ]:
# Features, die wir für die Modellierung nutzen
feature_cols = [c for c in ['fin_sqft', 'year_built', 'bedrooms', 'bathrooms',
                              'lotsize', 'stories', 'style', 'extwall', 'zip_code'] if c in df.columns]
target_col = price_col

df_model = df[feature_cols + [target_col]].dropna(subset=[target_col])
df_model.to_csv('../data/property_sales_clean.csv', index=False)

print(f'Gespeichert: {len(df_model):,} Zeilen, {len(feature_cols)} Features')
print('Features:', feature_cols)
df_model.head()